#### OpenAI SDK

In [ ]:
from openai import OpenAI
from pypdf import PdfReader
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key = os.getenv("openai_key"))

reader = PdfReader("./ejobIndia.pdf")
pdf_content = ""

for page in reader.pages:
    pdf_content += page.extract_text()

while True:
    user_input = input("Agent: ask about EjobIndia? ")
    if user_input.lower() == "exit":
        print("Agent: Cya!")
        exit(0)
    
    responses = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = [
            {
                "role" : "system",
                "content" : '''
                -You are a PDF Summarizer AI
                -You can only answer from provided PDF context
                -Apart from that Please Reply I dont Know anything.
                '''
            },
            {   #yaha par vi likh sakte ho ek sath
                "role" : "user",
                "content" : f'''
                -PDF Content:{pdf_content}
                -User Question :{user_input}
                -Answer only from Given PDF Content
                '''
            }
        ]
    )
    
    message = responses.choices[0].message.content
    print("Agent: ", message)

#### Gemini using Langchain

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pypdf import PdfReader
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(
    api_key = os.getenv("gemini_key"),
    model = "gemini-3.6-flash"
)

# Reading PDF from main folder
reader = PdfReader("./machine_learning.pdf")
pdf_content: str = ""

for page in reader.pages:
    pdf_content += page.extract_text()

prompt = PromptTemplate(
    template = """
    You are a PDF Summarizer AI.
    You can only answer from the provided PDF context.
    If the answer is not present in the PDF, reply:
    "I don't know anything about that."
    PDF Content:{_content}
    User Question:{_input}
    Answer only from the given PDF content.""",
    # input_variables= ["_content", "_input"] # modern LangChain automatically infers the variable names
)

while True:
    user_input = input("Agent: ask me about Machine Learning? ")

    if user_input.lower() == "exit":
        print("Agent: Bye Bye")
        exit(0)
    
    _prompt = prompt.invoke({
        "_content" : pdf_content,
        "_input" : user_input
    })
    
    responses = llm.invoke(_prompt)
    print("Agent: ", responses.content[0]['text'])
    
    

#### Gemini SDK + Streamlit

In [ ]:
import streamlit as st
from google import genai
from pypdf import PdfReader

from dotenv import load_dotenv
import os

# 1. PAGE CONFIG
st.set_page_config(
    page_title="PDF Summarizer Agent", page_icon="🤖", layout="centered"
)

# 2. CUSTOM CSS
st.html("""
<style>
    .stApp {
        background:
            radial-gradient(circle at top left, #1e1b4b, transparent 35%),
            radial-gradient(circle at bottom right, #172554, transparent 35%),
            #0f172a;
    }

    .block-container {
        max-width: 850px;
        padding-top: 40px;
    }

    /* HERO */
    .hero {
        text-align: center;
        padding: 30px 20px 35px;
    }

    .hero-icon {
        font-size: 60px;
        margin-bottom: 8px;
    }

    .hero-title {
        font-size: 48px;
        font-weight: 800;
        color: #a78bfa;
        margin-bottom: 8px;
    }

    .hero-subtitle {
        color: #94a3b8;
        font-size: 16px;
        line-height: 1.6;
    }

    /* INPUT TITLE */
    .section-title {
        font-size: 22px;
        font-weight: 700;
        color: #f8fafc;
        margin-bottom: 12px;
    }

    /* BUTTON */
    .stButton > button {
        width: 100%;
        height: 50px;
        border-radius: 12px;
        border: none;
        background: linear-gradient(
            90deg,
            #6366f1,
            #8b5cf6
        );
        color: white;
        font-size: 17px;
        font-weight: 700;
    }

    .stButton > button:hover {
        background: linear-gradient(
            90deg,
            #4f46e5,
            #7c3aed
        );
        color: white;
    }

    /* RESPONSE CARD */
    .response-card {
        margin-top: 20px;
        padding: 20px;
        border-radius: 16px;
        background: rgba(16, 185, 129, 0.08);
        border: 1px solid rgba(16, 185, 129, 0.25);
    }
    .response-title {
        color: #6ee7b7;
        font-size: 20px;
        font-weight: 700;
    }
</style>
""")

load_dotenv()

# 3. INITIALIZE GEMINI CLIENT
client = genai.Client(api_key=os.getenv("gemini_key"))

# 4. Read PDF content
reader = PdfReader("./renewable_energy.pdf")
pdf_content = ""

for page in reader.pages:
    pdf_content += page.extract_text()


# 5. HERO SECTION
st.html("""
<div class="hero">
    <div class="hero-icon">📄</div>
    <div class="hero-title">PDF Summarizer Agent</div>
    <div class="hero-subtitle">
        AI-powered document assistant<br>
        Ask questions about Renewable Energy document.
    </div>
</div>
""")

# 6. QUESTION INPUT
st.html("""
<div class="section-title">
    💬 Ask your PDF
</div>
""")

user_input = st.text_area(
    "Question",
    placeholder="""
        Try asking something like:
        • What is Renewable Energy?
        • Explain the technologies mentioned.
        • List key points mentioned such as advantages, challenges.""",
    height=160,
    label_visibility="collapsed",
)

send_btn = st.button("🚀 Ask Agent")

# 7. AGENT EXECUTION
if send_btn:
    if not user_input.strip():
        st.warning("Please enter a question first.")
    elif not pdf_content:
        st.error("PDF content is missing or could not be loaded.")
    else:
        with st.spinner("🧠 Agent is thinking..."):
            prompt = f"""
            PDF Content:{pdf_content}
            User Question:{user_input}"""
            
            system_instruction = """
            You are a PDF Summarizer AI.
            You can only answer from the provided PDF context.
            If the answer is not present in the PDF, reply: "I don't know anything about that." """

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
                config={"system_instruction": system_instruction},
            )

        st.html("""
        <div class="response-card">
            <div class="response-title">
                ✨ Agent Response
            </div>
        </div>
        """)
        st.write(response.text)

# 8. FOOTER
st.html("""
<div style="
    text-align:center;
    color:#64748b;
    margin-top:40px;
    padding-bottom:20px;
    font-size:13px;
">
    Powered by Streamlit + Gemini AI 🚀
</div>
""")

#### OpenAI SDK + Streamlit

In [ ]:
import streamlit as st
from openai import OpenAI # not langchain 
from dotenv import load_dotenv
from pypdf import PdfReader
import os

# PAGE CONFIG
st.set_page_config(
    page_title="PDF Analyzer AI", page_icon="🤖", layout="centered"
)

# 2. CUSTOM CSS
st.html("""
<style>
    .stApp {
        background:
            radial-gradient(circle at top left, #1e1b4b, transparent 35%),
            radial-gradient(circle at bottom right, #172554, transparent 35%),
            #0f172a;
    }

    .block-container {
        max-width: 850px;
        padding-top: 40px;
    }

    /* HERO */
    .hero {
        text-align: center;
        padding: 30px 20px 35px;
    }

    .hero-icon {
        font-size: 60px;
        margin-bottom: 8px;
    }

    .hero-title {
        font-size: 48px;
        font-weight: 800;
        color: #a78bfa;
        margin-bottom: 8px;
    }

    .hero-subtitle {
        color: #94a3b8;
        font-size: 16px;
        line-height: 1.6;
    }

    /* INPUT TITLE */
    .section-title {
        font-size: 22px;
        font-weight: 700;
        color: #f8fafc;
        margin-bottom: 12px;
    }

    /* BUTTON */
    .stButton > button {
        width: 100%;
        height: 50px;
        border-radius: 12px;
        border: none;
        background: linear-gradient(
            90deg,
            #6366f1,
            #8b5cf6
        );
        color: white;
        font-size: 17px;
        font-weight: 700;
    }

    .stButton > button:hover {
        background: linear-gradient(
            90deg,
            #4f46e5,
            #7c3aed
        );
        color: white;
    }

    /* RESPONSE CARD */
    .response-card {
        margin-top: 20px;
        padding: 20px;
        border-radius: 16px;
        background: rgba(16, 185, 129, 0.08);
        border: 1px solid rgba(16, 185, 129, 0.25);
    }
    .response-title {
        color: #6ee7b7;
        font-size: 20px;
        font-weight: 700;
    }
</style>
""")

load_dotenv()

# Connect to OpenAI LLM
client = OpenAI(api_key=os.getenv("openai_key"))

# Reading PDF
reader = PdfReader("./networking.pdf")
pdf_content = ""

for page in reader.pages:
    pdf_content += page.extract_text()

# 5. HERO SECTION
st.html("""
<div class="hero">
    <div class="hero-icon">📄</div>
    <div class="hero-title">PDF Summarizer Agent</div>
    <div class="hero-subtitle">
        AI-powered document assistant<br>
        Ask questions about Renewable Energy document.
    </div>
</div>
""")

# 6. QUESTION INPUT
st.html("""
<div class="section-title">
    💬 Ask your PDF
</div>
""")

user_input = st.text_area(
    "Question",
    placeholder="""
        Try asking something like:
        • What is Renewable Energy?
        • Explain the technologies mentioned.
        • List key points mentioned such as advantages, challenges.""",
    height=160,
    label_visibility="collapsed",
)

send_btn = st.button("🚀 Ask Agent")


# 7. AGENT EXECUTION
if send_btn:
    if not user_input.strip():
        st.warning("Please enter a question first.")
    elif not pdf_content:
        st.error("PDF content is missing or could not be loaded.")
    else:
        with st.spinner("🧠 Agent is thinking..."):
            prompt = f"""
            PDF Content:{pdf_content}
            User Question:{user_input}"""
            
            system_instruction = """
            You are a PDF Summarizer AI.
            You can only answer from the provided PDF context.
            If the answer is not present in the PDF, reply: "I don't know anything about that." """

            response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {
                        "role": "system",
                        "content": """
                        You are a PDF Summarizer AI.
                        You can only answer from the provided PDF context.
                        If the answer is not available in the PDF,
                        reply: I don't know."""},
                    {
                        "role": "user",
                        "content": f"""
                        PDF Content: {pdf_content}
                        User Question: {user_input}
                        Answer only from the given PDF content."""}
                ]
            )
            
            st.html("""
            <div class="response-card">
                <div class="response-title">
                    ✨ Agent Response
                </div>
            </div>
            """)
            message = response.choices[0].message.content
            st.write(message)

# 8. FOOTER
st.html("""
<div style="
    text-align:center;
    color:#64748b;
    margin-top:40px;
    padding-bottom:20px;
    font-size:13px;
">
    Powered by Streamlit + OpenAI 🚀
</div>
""")